# Training the 5-Channel DistilBERT Architecture
This notebook focuses on training the custom 5-Channel DistilBERT model.

The architecture is designed to process a main text body alongside its corresponding rule, plus positive and negative examples, to learn the nuances of rule violations.

## Data Acquisition
The dataset is retrieved directly from the Kaggle competition using the Kaggle API.

To reproduce this environment:
1. Upload your `kaggle.json` API token.
2. Run the following commands to download and extract the data:

```bash
# !pip install kaggle
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c jigsaw-agile-community-rules
# !unzip jigsaw-agile-community-rules.zip -d data

## 1. Environment Setup and Data Loading
We initialize the environment, handle dependencies, and prepare the stop words for text cleaning.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import nltk
from nltk.corpus import stopwords
import os

# Download and configure stop words
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english')) - {"not", "no", "never", "none", "neither", "nor", "against"}

def clean_stop_words(text):
    if pd.isna(text): return ""
    return " ".join([word for word in str(text).split() if word.lower() not in stop_words])

## 2. Dataset and Model Definition
We define a ContrastiveDataset to manage the five input channels and the FiveChannelDistilBert class which aggregates these channels into a single decision layer.

In [ ]:
class ContrastiveDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def tokenize_pair(self, text_a, text_b):
        return self.tokenizer(
            clean_stop_words(text_a),
            clean_stop_words(text_b),
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rule = row['rule']
        return {
            "body": self.tokenize_pair(row['body'], rule),
            "pos1": self.tokenize_pair(row['positive_example_1'], rule),
            "pos2": self.tokenize_pair(row['positive_example_2'], rule),
            "neg1": self.tokenize_pair(row['negative_example_1'], rule),
            "neg2": self.tokenize_pair(row['negative_example_2'], rule),
            "label": torch.tensor(row['rule_violation'], dtype=torch.float)
        }

class FiveChannelDistilBert(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.distilbert = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(5, 1)

    def get_score(self, inputs):
        input_ids = inputs['input_ids'].squeeze(1).to(device)
        attention_mask = inputs['attention_mask'].squeeze(1).to(device)
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        # Using [CLS] token representation
        return torch.mean(outputs.last_hidden_state[:, 0, :], dim=1)

    def forward(self, batch):
        channels = ['body', 'pos1', 'pos2', 'neg1', 'neg2']
        combined = torch.stack([self.get_score(batch[k]) for k in channels], dim=1)
        return self.classifier(combined).squeeze(-1)

## 3. Training Pipeline
Initialization of the model, optimizer, and the training loop. We utilize BCEWithLogitsLoss for binary classification across the 5 aggregated channels.

In [ ]:
# Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = FiveChannelDistilBert(model_name).to(device)

# Load data
train_df = pd.read_csv("data/train_processed.csv")
train_loader = DataLoader(ContrastiveDataset(train_df, tokenizer), batch_size=8, shuffle=True)

# Optimization settings
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

# Training loop
model.train()
for epoch in range(3):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for batch in progress_bar:
        optimizer.zero_grad()
        logits = model(batch)
        loss = criterion(logits, batch['label'].to(device))
        loss.backward()
        optimizer.step()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

## 4. Model Export and Artifact Storage
After training, we save the base transformer configuration and the custom head weights separately for future inference and feature extraction.

In [ ]:
# Define export path
save_path = "distilbert-5channel-final"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Save Transformer components and local weights
model.distilbert.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
torch.save(model.state_dict(), f"{save_path}/five_channel_final_weights.pt")